# 01 - Exploratory data analysis

What does the chest X-ray dataset look like? We check class balance, image sizes and brightness, and look at a few examples before modelling anything.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make the project's `src` package importable from inside notebooks/
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), Path.cwd())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src import config
from src.utils import find_dataset_root, count_images_per_class

# Point this at data/dummy to try the notebook without the real dataset
DATA_DIR = config.DATA_DIR
ROOT_DATA = find_dataset_root(DATA_DIR)
print("Dataset root:", ROOT_DATA)

## Class balance per split

Pneumonia is the majority class in `train/`, so accuracy alone would be misleading - we use class weights, recall and AUC later.

In [ ]:
splits = [s for s in ("train", "val", "test") if (ROOT_DATA / s).is_dir()]
counts = {s: count_images_per_class(ROOT_DATA / s) for s in splits}
for s, c in counts.items():
    print(f"{s:<5}", c)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(splits)); w = 0.38
for i, cls in enumerate(config.CLASS_NAMES):
    ax.bar(x + (i - 0.5) * w, [counts[s][cls] for s in splits], w, label=cls)
ax.set_xticks(x, splits); ax.set_ylabel("images"); ax.set_title("Images per split and class"); ax.legend()
plt.show()

## Sample images

In [ ]:
def sample_paths(split, cls, n):
    files = sorted((ROOT_DATA / split / cls).glob("*"))
    files = [f for f in files if f.suffix.lower() in config.IMG_EXTS]
    return files[:n]

fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for r, cls in enumerate(config.CLASS_NAMES):
    for c, path in enumerate(sample_paths("train", cls, 5)):
        axes[r, c].imshow(Image.open(path).convert("L"), cmap="gray")
        axes[r, c].set_title(cls); axes[r, c].axis("off")
plt.tight_layout(); plt.show()

## Image sizes and brightness

X-rays come in many different resolutions, which is why every image is resized to a fixed square before training.

In [ ]:
rows = []
for cls in config.CLASS_NAMES:
    for path in sample_paths("train", cls, 150):
        with Image.open(path) as im:
            w, h = im.size
            gray = np.asarray(im.convert("L"), dtype="float32")
        rows.append((cls, w, h, gray.mean(), gray.std()))

import pandas as pd
df = pd.DataFrame(rows, columns=["class", "width", "height", "mean_intensity", "std_intensity"])
display(df.groupby("class").agg(["mean", "min", "max"]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for cls in config.CLASS_NAMES:
    axes[0].hist(df[df["class"] == cls]["mean_intensity"], bins=25, alpha=0.6, label=cls)
    axes[1].scatter(df[df["class"] == cls]["width"], df[df["class"] == cls]["height"], alpha=0.5, label=cls)
axes[0].set_title("Mean pixel intensity"); axes[0].legend()
axes[1].set_title("Image width vs height"); axes[1].set_xlabel("width"); axes[1].set_ylabel("height")
plt.tight_layout(); plt.show()

## Takeaways to carry forward

- Training data is imbalanced -> use class weights and report recall / F1 / ROC-AUC, not just accuracy.
- The official `val/` folder is tiny, so `src/data.py` splits a validation set out of `train/` instead.
- Image sizes vary a lot -> resize to 224 x 224.
- Intensity differs between images -> the model must not rely on brightness alone; Grad-CAM (notebook 05) helps us check.